In [ ]:
import pandas as pd

In [1]:
df_consumer = pd.read_parquet('mlc/training_data/consumer_data.parquet')
df_transaction = pd.read_parquet('mlc/training_data/transactions.parquet')


NameError: name 'pd' is not defined

In [ ]:
df_consumer['evaluation_date'] = pd.to_datetime(df_consumer['evaluation_date'])
df_transaction['posted_date'] = pd.to_datetime(df_transaction['posted_date'])

In [ ]:
df_consumer.info()
df_transaction.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   evaluation_date     16000 non-null  datetime64[ns]
 1   FPF_TARGET          16000 non-null  float64       
 2   total_balance       15999 non-null  float64       
 3   masked_consumer_id  16000 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 625.0+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 17738083 entries, 2715977 to 1072784
Data columns (total 5 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   masked_consumer_id     object        
 1   posted_date            datetime64[ns]
 2   amount                 float64       
 3   category               float64       
 4   masked_transaction_id  object        
dtypes: datetime64[ns](1), float64(2), object(2)
memory usage: 812.0+ MB


In [ ]:
df_consumer.isnull().sum()

evaluation_date       0
FPF_TARGET            0
total_balance         1
masked_consumer_id    0
dtype: int64

In [ ]:
df_transaction.isnull().sum()

masked_consumer_id         0
posted_date              121
amount                   121
category                 121
masked_transaction_id      0
dtype: int64

In [ ]:
df_transaction['masked_consumer_id'].nunique()

16000

In [ ]:
df_consumer[df_consumer['total_balance'].isnull()]

,evaluation_date,FPF_TARGET,total_balance,masked_consumer_id
777,2022-10-09,0.0,NaN,C03100778


In [ ]:
df_transaction[df_transaction['masked_consumer_id'] == 'C03100778']

,masked_consumer_id,posted_date,amount,category,masked_transaction_id
958635,C03100778,NaT,NaN,NaN,C03T0958635


In [ ]:
df_consumer_merged = pd.merge(df_consumer, df_transaction, how='left', on='masked_consumer_id')
df_consumer_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17738083 entries, 0 to 17738082
Data columns (total 8 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   evaluation_date        datetime64[ns]
 1   FPF_TARGET             float64       
 2   total_balance          float64       
 3   masked_consumer_id     object        
 4   posted_date            datetime64[ns]
 5   amount                 float64       
 6   category               float64       
 7   masked_transaction_id  object        
dtypes: datetime64[ns](2), float64(4), object(2)
memory usage: 1.1+ GB


In [ ]:
df_consumer_merged.head()

,evaluation_date,FPF_TARGET,total_balance,masked_consumer_id,posted_date,amount,category,masked_transaction_id
0,2021-08-26,0.0,1380.84,C01100001,2021-06-29,926.89,4.0,C01T00
1,2021-08-26,0.0,1380.84,C01100001,2020-11-16,300.00,0.0,C01T01
2,2021-08-26,0.0,1380.84,C01100001,2020-11-25,500.00,0.0,C01T02
3,2021-08-26,0.0,1380.84,C01100001,2020-09-03,1000.00,0.0,C01T03
4,2021-08-26,0.0,1380.84,C01100001,2020-11-09,200.00,0.0,C01T04


In [ ]:
df_consumer_merged[df_consumer_merged['amount'].isnull()]

,evaluation_date,FPF_TARGET,total_balance,masked_consumer_id,posted_date,amount,category,masked_transaction_id
7492610,2022-09-12,1.0,0.00,C03100010,NaT,NaN,NaN,C03T012247
7559314,2022-09-11,0.0,228.02,C03100076,NaT,NaN,NaN,C03T0107292
7559504,2022-09-12,0.0,346.83,C03100079,NaT,NaN,NaN,C03T0107784
7596107,2022-09-10,0.0,0.00,C03100121,NaT,NaN,NaN,C03T0160281
7598862,2022-08-24,0.0,7696.30,C03100127,NaT,NaN,NaN,C03T0166762
...,...,...,...,...,...,...,...,...
12202288,2023-02-06,0.0,0.00,C03104772,NaT,NaN,NaN,C03T05922506
12222190,2023-02-04,0.0,0.00,C03104797,NaT,NaN,NaN,C03T05948810
12363842,2023-01-20,0.0,0.00,C03104921,NaT,NaN,NaN,C03T06120226
12397226,2023-01-27,0.0,0.00,C03104959,NaT,NaN,NaN,C03T06162426


In [ ]:
df_filtered = df_consumer_merged[df_consumer_merged['evaluation_date'] > df_consumer_merged['posted_date']]
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17578653 entries, 0 to 17738082
Data columns (total 8 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   evaluation_date        datetime64[ns]
 1   FPF_TARGET             float64       
 2   total_balance          float64       
 3   masked_consumer_id     object        
 4   posted_date            datetime64[ns]
 5   amount                 float64       
 6   category               float64       
 7   masked_transaction_id  object        
dtypes: datetime64[ns](2), float64(4), object(2)
memory usage: 1.2+ GB


In [ ]:
df_consumer_merged[df_consumer_merged['evaluation_date'] == df_consumer_merged['posted_date']].nunique()

evaluation_date            716
FPF_TARGET                   2
total_balance             8448
masked_consumer_id        8956
posted_date                716
amount                   13689
category                    35
masked_transaction_id    57577
dtype: int64

In [ ]:
# Create pipeline: create features from transcations, merge with consumer table, train test split, imputation, baseline model (logistic regression), and evaluate model performance using roc_auc. Here are some features to create:
# Aggregate statistics (e.g., mean, median, min, max, sum):

# Monthly/weekly average balance.

# Frequency and magnitude of debits and credits.

# Category-based spending patterns:

# Recent activity features (e.g., recent 30-day balance changes):

# Cashflow volatility:

# Compute coefficient of variation (std/mean) of monthly cashflows.

# Ratio of credits to debits:

# Merge all derived features into a single feature set:

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from datetime import timedelta
from xgboost import XGBClassifier

In [ ]:
consumer_df = pd.read_parquet('mlc/training_data/consumer_data.parquet')
transactions_df = pd.read_parquet('mlc/training_data/transactions.parquet')

In [ ]:
consumer_df['evaluation_date'] = pd.to_datetime(consumer_df['evaluation_date'])
transactions_df['posted_date'] = pd.to_datetime(transactions_df['posted_date'])

In [ ]:
# Step 1: Filter transactions to only those before or on the evaluation date
transactions_df = transactions_df.merge(
    consumer_df[['masked_consumer_id', 'evaluation_date']],
    on='masked_consumer_id',
    how='left'
)
transactions_df = transactions_df[transactions_df['posted_date'] < transactions_df['evaluation_date']]

In [ ]:
# Step 2: Feature Engineering
# Create debit/credit indicators
transactions_df['is_credit'] = transactions_df['amount'] > 0
transactions_df['is_debit'] = transactions_df['amount'] < 0

# Compute aggregate features
agg = transactions_df.groupby('masked_consumer_id').agg(
    total_amount=('amount', 'sum'),
    mean_amount=('amount', 'mean'),
    std_amount=('amount', 'std'),
    min_amount=('amount', 'min'),
    max_amount=('amount', 'max'),
    median_amount=('amount', 'median'),
    transaction_count=('amount', 'count'),
    credit_sum=('is_credit', lambda x: transactions_df.loc[x.index, 'amount'][x].sum()),
    debit_sum=('is_debit', lambda x: abs(transactions_df.loc[x.index, 'amount'][x].sum()))
)

# Ratio of credits to debits
agg['credit_debit_ratio'] = agg['credit_sum'] / agg['debit_sum'].replace(0, np.nan)
agg['credit_debit_ratio'] = agg['credit_debit_ratio'].fillna(0)

# Recent 30-day transaction features
transactions_df['days_before_eval'] = (transactions_df['evaluation_date'] - transactions_df['posted_date']).dt.days
recent_df = transactions_df[transactions_df['days_before_eval'] <= 30].groupby('masked_consumer_id').agg(
    recent_sum=('amount', 'sum'),
    recent_count=('amount', 'count')
)

# Monthly cashflow volatility: std / mean
transactions_df['month'] = transactions_df['posted_date'].dt.to_period('M')
monthly_cashflow = transactions_df.groupby(['masked_consumer_id', 'month'])['amount'].sum().reset_index()
monthly_stats = monthly_cashflow.groupby('masked_consumer_id')['amount'].agg(['mean', 'std']).rename(
    columns={'mean': 'monthly_mean', 'std': 'monthly_std'}
)
monthly_stats['monthly_cv'] = monthly_stats['monthly_std'] / monthly_stats['monthly_mean'].replace(0, np.nan)
monthly_stats = monthly_stats.fillna(0)

# Merge all features
features_df = consumer_df.set_index('masked_consumer_id') \
    .join(agg, how='left') \
    .join(recent_df, how='left') \
    .join(monthly_stats, how='left') \
    .fillna(0)

In [ ]:
features_df

,evaluation_date,FPF_TARGET,total_balance,total_amount,mean_amount,std_amount,min_amount,max_amount,median_amount,transaction_count,credit_sum,debit_sum,credit_debit_ratio,recent_sum,recent_count,monthly_mean,monthly_std,monthly_cv
masked_consumer_id,,,,,,,,,,,,,,,,,,
C01100001,2021-08-26,0.0,1380.84,-8111.42,-3.051701,645.540740,-8000.00,8483.00,-25.380,2658.0,276864.26,284975.68,0.971536,1045.13,189.0,-623.955385,4616.419813,-7.398638
C01100002,2022-08-02,0.0,20163.90,-9490.73,-3.813070,910.144154,-19150.00,20000.00,-25.290,2489.0,285876.77,295367.50,0.967868,-940.59,175.0,-730.056154,5862.877315,-8.030721
C01100003,2021-03-04,0.0,3986.25,2137.77,3.224389,748.056684,-4310.28,4409.40,-21.350,663.0,84178.75,82040.98,1.026057,-554.21,62.0,164.443846,1092.197234,6.641764
C01100004,2022-11-19,0.0,5956.03,-29837.87,-27.525710,414.375001,-1800.00,1748.00,-27.555,1084.0,97426.62,127264.49,0.765544,-1637.19,96.0,-2295.220769,1769.698164,-0.771036
C01100005,2021-11-21,0.0,29421.10,-9756.37,-4.044930,1314.772339,-6987.07,39915.16,-25.080,2412.0,404922.85,414679.22,0.976472,3197.70,193.0,-750.490000,15567.716374,-20.743403
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C04104996,2023-03-31,0.0,1412.80,-6255.18,-8.134174,240.285238,-2944.86,2000.00,-19.990,769.0,34087.79,40342.97,0.844950,-1799.91,89.0,-521.265000,1446.734327,-2.775430
C04104997,2023-03-31,0.0,-439.21,-821.02,-0.715798,384.276675,-1539.41,4559.67,-17.120,1147.0,51369.16,52190.18,0.984269,-2990.67,90.0,-68.418333,1251.100938,-18.286048
C04104998,2023-03-31,0.0,84.53,-1823.87,-0.998834,432.937104,-1000.00,6199.20,-20.535,1826.0,93807.66,95631.53,0.980928,-1516.99,150.0,-151.989167,1738.442369,-11.437936


In [ ]:
# Step 3: Train-Test Split
X = features_df.drop(columns=['FPF_TARGET', 'evaluation_date'])
y = features_df['FPF_TARGET']
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [ ]:
# Step 4: Pipeline with Imputation and Logistic Regression
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('model', LogisticRegression(max_iter=1000)),
    # ('model', XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42))
])
pipeline.fit(X_train, y_train)

NameError: name 'Pipeline' is not defined

In [ ]:
# Step 5: Evaluate Performance
y_train_pred = pipeline.predict_proba(X_train)[:, 1]
y_val_pred = pipeline.predict_proba(X_val)[:, 1]
train_auc = roc_auc_score(y_train, y_train_pred)
val_auc = roc_auc_score(y_val, y_val_pred)
print(f"Train AUC: {train_auc:.4f}, Validation AUC: {val_auc:.4f}")

Train AUC: 0.9623, Validation AUC: 0.8596
